# 🏦 Lender's Club – Credit Risk Modeling & Stress Testing

---

**Author:** Om  
**Date:** March 2026  
**Tools Used:** Python, Pandas, Scikit-learn, XGBoost, Streamlit, Plotly  

---

## Project Summary

This project builds an **end-to-end credit risk model** using real-world Lending Club data to predict which borrowers are likely to **default on their loans**.

### 🔑 Key Highlights:
- Cleaned & preprocessed **30,000+ loan records**
- Applied **WOE (Weight of Evidence)** and **IV (Information Value)** for feature engineering
- Trained & compared **Logistic Regression, Random Forest, and XGBoost** models
- Performed **stress testing** under simulated recession conditions
- Built an interactive **Streamlit dashboard** for real-time portfolio monitoring

### 📂 Project Structure:
| File | Description |
| --- | --- |
| `Lenders_Club_Project_Structured.ipynb` | This notebook — full analysis |
| `final_dataset.csv` | Model output with risk scores |
| `dashboard.py` | Interactive Streamlit dashboard |

---

In [112]:
import pandas as pd
import numpy as np

## 1. Data Loading & Initial Exploration

We start by loading the **Lending Club loan dataset** and doing a quick look at its size, columns, and structure.

In [113]:
df = pd.read_csv('C:\\Users\om\Project Resume\project_sample.csv')
df

,loan_amnt,funded_amnt,funded_amnt_inv,term,int_rate,installment,grade,sub_grade,emp_title,emp_length,...,hardship_payoff_balance_amount,hardship_last_payment_amount,disbursement_method,debt_settlement_flag,debt_settlement_flag_date,settlement_status,settlement_date,settlement_amount,settlement_percentage,settlement_term
0,35000,35000,35000.0,36 months,12.12,1164.51,B,B3,Legacy Physicians Group,< 1 year,...,NaN,NaN,Cash,N,NaN,NaN,NaN,NaN,NaN,NaN
1,30000,30000,30000.0,60 months,10.75,648.54,B,B4,Director of nursing,2 years,...,NaN,NaN,Cash,N,NaN,NaN,NaN,NaN,NaN,NaN
2,15000,15000,15000.0,36 months,7.49,466.53,A,A4,Partner,5 years,...,NaN,NaN,Cash,N,NaN,NaN,NaN,NaN,NaN,NaN
3,24000,24000,24000.0,60 months,21.15,651.31,E,E2,Einstein Bros. Bagels,5 years,...,NaN,NaN,Cash,N,NaN,NaN,NaN,NaN,NaN,NaN
4,14400,14400,14400.0,36 months,8.59,455.18,A,A5,Nurse practitioner,3 years,...,NaN,NaN,Cash,N,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
99995,4000,4000,4000.0,36 months,13.49,135.73,C,C2,Center Based Special Education Teacher,5 years,...,NaN,NaN,Cash,N,NaN,NaN,NaN,NaN,NaN,NaN
99996,35000,35000,35000.0,60 months,15.59,843.53,D,D1,Senior Director,7 years,...,NaN,NaN,Cash,N,NaN,NaN,NaN,NaN,NaN,NaN
99997,20000,20000,20000.0,60 months,15.49,480.96,C,C4,Registered Dental Hygienist,2 years,...,NaN,NaN,DirectPay,N,NaN,NaN,NaN,NaN,NaN,NaN
99998,16000,16000,16000.0,36 months,11.06,524.28,B,B3,Cash Management Manager,10+ years,...,NaN,NaN,Cash,N,NaN,NaN,NaN,NaN,NaN,NaN


In [114]:
df

,loan_amnt,funded_amnt,funded_amnt_inv,term,int_rate,installment,grade,sub_grade,emp_title,emp_length,...,hardship_payoff_balance_amount,hardship_last_payment_amount,disbursement_method,debt_settlement_flag,debt_settlement_flag_date,settlement_status,settlement_date,settlement_amount,settlement_percentage,settlement_term
0,35000,35000,35000.0,36 months,12.12,1164.51,B,B3,Legacy Physicians Group,< 1 year,...,NaN,NaN,Cash,N,NaN,NaN,NaN,NaN,NaN,NaN
1,30000,30000,30000.0,60 months,10.75,648.54,B,B4,Director of nursing,2 years,...,NaN,NaN,Cash,N,NaN,NaN,NaN,NaN,NaN,NaN
2,15000,15000,15000.0,36 months,7.49,466.53,A,A4,Partner,5 years,...,NaN,NaN,Cash,N,NaN,NaN,NaN,NaN,NaN,NaN
3,24000,24000,24000.0,60 months,21.15,651.31,E,E2,Einstein Bros. Bagels,5 years,...,NaN,NaN,Cash,N,NaN,NaN,NaN,NaN,NaN,NaN
4,14400,14400,14400.0,36 months,8.59,455.18,A,A5,Nurse practitioner,3 years,...,NaN,NaN,Cash,N,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
99995,4000,4000,4000.0,36 months,13.49,135.73,C,C2,Center Based Special Education Teacher,5 years,...,NaN,NaN,Cash,N,NaN,NaN,NaN,NaN,NaN,NaN
99996,35000,35000,35000.0,60 months,15.59,843.53,D,D1,Senior Director,7 years,...,NaN,NaN,Cash,N,NaN,NaN,NaN,NaN,NaN,NaN
99997,20000,20000,20000.0,60 months,15.49,480.96,C,C4,Registered Dental Hygienist,2 years,...,NaN,NaN,DirectPay,N,NaN,NaN,NaN,NaN,NaN,NaN
99998,16000,16000,16000.0,36 months,11.06,524.28,B,B3,Cash Management Manager,10+ years,...,NaN,NaN,Cash,N,NaN,NaN,NaN,NaN,NaN,NaN


## 2. Data Cleaning & Preprocessing

Raw data is messy. In this section we:
- Identify columns that are **entirely NaN** (empty) and can be dropped
- Fill remaining missing values with `'Missing'` as a placeholder
- Save the cleaned version for further processing

> 💡 **Why fill NaN with 'Missing'?** — In credit risk, missingness itself can be informative. A borrower not providing data might indicate higher risk.

In [115]:
nan_cols = df.columns[df.isna().all()]
nan_cols

Index([], dtype='object')

In [116]:
df.to_csv('project_sample.csv', index=False)

In [141]:
df = df.fillna('Missing')
df

,loan_amnt,funded_amnt,funded_amnt_inv,term,int_rate,installment,grade,sub_grade,emp_title,emp_length,...,sec_app_mort_acc,sec_app_open_acc,sec_app_revol_util,sec_app_open_act_il,sec_app_num_rev_accts,sec_app_chargeoff_within_12_mths,sec_app_collections_12_mths_ex_med,sec_app_mths_since_last_major_derog,disbursement_method,target
0,35000,35000,35000.0,36 months,12.12,1164.51,B,B3,Legacy Physicians Group,< 1 year,...,Missing,Missing,Missing,Missing,Missing,Missing,Missing,Missing,Cash,0
1,30000,30000,30000.0,60 months,10.75,648.54,B,B4,Director of nursing,2 years,...,Missing,Missing,Missing,Missing,Missing,Missing,Missing,Missing,Cash,0
2,15000,15000,15000.0,36 months,7.49,466.53,A,A4,Partner,5 years,...,Missing,Missing,Missing,Missing,Missing,Missing,Missing,Missing,Cash,0
3,24000,24000,24000.0,60 months,21.15,651.31,E,E2,Einstein Bros. Bagels,5 years,...,Missing,Missing,Missing,Missing,Missing,Missing,Missing,Missing,Cash,0
4,14400,14400,14400.0,36 months,8.59,455.18,A,A5,Nurse practitioner,3 years,...,Missing,Missing,Missing,Missing,Missing,Missing,Missing,Missing,Cash,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
99995,4000,4000,4000.0,36 months,13.49,135.73,C,C2,Center Based Special Education Teacher,5 years,...,Missing,Missing,Missing,Missing,Missing,Missing,Missing,Missing,Cash,0
99996,35000,35000,35000.0,60 months,15.59,843.53,D,D1,Senior Director,7 years,...,Missing,Missing,Missing,Missing,Missing,Missing,Missing,Missing,Cash,0
99997,20000,20000,20000.0,60 months,15.49,480.96,C,C4,Registered Dental Hygienist,2 years,...,Missing,Missing,Missing,Missing,Missing,Missing,Missing,Missing,DirectPay,0
99998,16000,16000,16000.0,36 months,11.06,524.28,B,B3,Cash Management Manager,10+ years,...,Missing,Missing,Missing,Missing,Missing,Missing,Missing,Missing,Cash,0


In [118]:
for v in df.columns:
    print(v)

loan_amnt
funded_amnt
funded_amnt_inv
term
int_rate
installment
grade
sub_grade
emp_title
emp_length
home_ownership
annual_inc
verification_status
issue_d
loan_status
pymnt_plan
desc
purpose
title
zip_code
addr_state
dti
delinq_2yrs
earliest_cr_line
inq_last_6mths
mths_since_last_delinq
mths_since_last_record
open_acc
pub_rec
revol_bal
revol_util
total_acc
initial_list_status
out_prncp
out_prncp_inv
total_pymnt
total_pymnt_inv
total_rec_prncp
total_rec_int
total_rec_late_fee
recoveries
collection_recovery_fee
last_pymnt_d
last_pymnt_amnt
next_pymnt_d
last_credit_pull_d
collections_12_mths_ex_med
mths_since_last_major_derog
policy_code
application_type
annual_inc_joint
dti_joint
verification_status_joint
acc_now_delinq
tot_coll_amt
tot_cur_bal
open_acc_6m
open_act_il
open_il_12m
open_il_24m
mths_since_rcnt_il
total_bal_il
il_util
open_rv_12m
open_rv_24m
max_bal_bc
all_util
total_rev_hi_lim
inq_fi
total_cu_tl
inq_last_12m
acc_open_past_24mths
avg_cur_bal
bc_open_to_buy
bc_util
chargeoff_

In [119]:
df['loan_status'].unique()

array(['Fully Paid', 'Current', 'Charged Off', 'In Grace Period',
       'Late (31-120 days)', 'Late (16-30 days)',
       'Does not meet the credit policy. Status:Charged Off',
       'Does not meet the credit policy. Status:Fully Paid', 'Default'],
      dtype=object)

## 3. Exploratory Data Analysis (EDA) & Feature Leakage Removal

### What is Feature Leakage?
Some columns contain information that **would not be available at the time of prediction** (e.g., total payments received). Including them would give the model an unfair advantage and produce misleading results in production.

### Steps in this section:
1. Remove **data leakage columns** (post-loan outcome variables)
2. Identify columns with few unique values (categorical candidates)
3. Identify columns with many unique values (numerical candidates)
4. Create the **target variable**: `1 = Charged Off (Defaulted)`, `0 = Fully Paid`
5. Encode categorical variables and bin numerical features

In [120]:
leakage_cols = [
    'total_pymnt','total_pymnt_inv','total_rec_prncp',
    'total_rec_int','total_rec_late_fee',
    'recoveries','collection_recovery_fee',
    

    'out_prncp','out_prncp_inv',
    
    'last_pymnt_d','last_pymnt_amnt','next_pymnt_d',
    
    'last_credit_pull_d',
    
    'settlement_status','settlement_date','settlement_amount',
    'settlement_percentage','settlement_term',
    
    'hardship_flag','hardship_type','hardship_reason',
    'hardship_status','deferral_term','hardship_amount',
    'hardship_start_date','hardship_end_date',
    'payment_plan_start_date','hardship_length',
    'hardship_dpd','hardship_loan_status',
    'orig_projected_additional_accrued_interest',
    'hardship_payoff_balance_amount','hardship_last_payment_amount',
    
    'debt_settlement_flag','debt_settlement_flag_date'
]

df = df.drop(columns=leakage_cols, errors='ignore')

In [121]:
unq = pd.DataFrame(df.nunique().reset_index())
unq

,index,0
0,loan_amnt,1428
1,funded_amnt,1428
2,funded_amnt_inv,1835
3,term,2
4,int_rate,570
...,...,...
102,sec_app_num_rev_accts,56
103,sec_app_chargeoff_within_12_mths,8
104,sec_app_collections_12_mths_ex_med,9
105,sec_app_mths_since_last_major_derog,99


In [122]:
unq = unq.rename(columns={0:'values'})
unq

,index,values
0,loan_amnt,1428
1,funded_amnt,1428
2,funded_amnt_inv,1835
3,term,2
4,int_rate,570
...,...,...
102,sec_app_num_rev_accts,56
103,sec_app_chargeoff_within_12_mths,8
104,sec_app_collections_12_mths_ex_med,9
105,sec_app_mths_since_last_major_derog,99


In [123]:
df2 = unq[unq['values']<11]
df2

,index,values
3,term,2
6,grade,7
10,home_ownership,6
12,verification_status,3
14,loan_status,9
15,pymnt_plan,2
32,initial_list_status,2
33,collections_12_mths_ex_med,8
35,policy_code,1
36,application_type,2


In [124]:
for v in df2['index']:
    print(v)

term
grade
home_ownership
verification_status
loan_status
pymnt_plan
initial_list_status
collections_12_mths_ex_med
policy_code
application_type
verification_status_joint
acc_now_delinq
chargeoff_within_12_mths
num_tl_120dpd_2m
num_tl_30dpd
pub_rec_bankruptcies
sec_app_inq_last_6mths
sec_app_chargeoff_within_12_mths
sec_app_collections_12_mths_ex_med
disbursement_method


In [125]:
df['target'] = df['loan_status'].apply(
    lambda x: 1 if x == 'Charged Off' else 0
)

In [252]:
cols = {'term','grade',
"home_ownership",
"verification_status","pymnt_plan",
"initial_list_status",
"collections_12_mths_ex_med",
"policy_code",
"application_type",
"verification_status_joint",
"acc_now_delinq",
"chargeoff_within_12_mths",
"num_tl_120dpd_2m",
"num_tl_30dpd",
"pub_rec_bankruptcies",
"sec_app_inq_last_6mths",
"sec_app_chargeoff_within_12_mths",
"sec_app_collections_12_mths_ex_med",
"disbursement_method",'sub_grade','purpose'}

final_list = []

for col in cols:
    
    temp = df[col].astype(object).fillna('Missing') 
    
    group = df.groupby(temp)["target"].agg(['count', 'sum'])
    group.columns = ['Total', 'Bad']

    group['Good'] = group['Total'] - group['Bad']

    total_good = group['Good'].sum()
    total_bad  = group['Bad'].sum()

    group['Good Pct'] = group['Good'] / total_good
    group['Bad Pct']  = group['Bad']  / total_bad

    group[['Good Pct', 'Bad Pct']] = group[['Good Pct', 'Bad Pct']].replace(0, 0.0001)

    group['WOE'] = np.log(group['Good Pct'] / group['Bad Pct'])
    group['IV']  = (group['Good Pct'] - group['Bad Pct']) * group['WOE']

    group['dimension'] = col
    group['category']  = group.index

    final_list.append(group.reset_index(drop=True))

df_iv = pd.concat(final_list, ignore_index=True)
df_iv

,Total,Bad,Good,Good Pct,Bad Pct,WOE,IV,dimension,category
0,3807,61,3746,0.042342,0.005291,2.079859,0.077062,sub_grade,A1
1,3079,61,3018,0.034113,0.005291,1.863765,0.053719,sub_grade,A2
2,3192,99,3093,0.034961,0.008586,1.404066,0.037032,sub_grade,A3
3,4334,168,4166,0.047089,0.014571,1.173036,0.038146,sub_grade,A4
4,4868,262,4606,0.052063,0.022723,0.829059,0.024324,sub_grade,A5
...,...,...,...,...,...,...,...,...,...
140,2,0,2,0.000023,0.000100,-1.486931,0.000115,sec_app_chargeoff_within_12_mths,7.0
141,1,0,1,0.000011,0.000100,-2.180078,0.000193,sec_app_chargeoff_within_12_mths,17.0
142,4664,171,4493,0.050786,0.014831,1.230901,0.044257,sec_app_chargeoff_within_12_mths,False
143,95209,11349,83860,0.947892,0.984302,-0.037692,0.001372,sec_app_chargeoff_within_12_mths,Missing


In [206]:
df3 = unq[unq['values']>11]
df3

,index,values
0,loan_amnt,1428
1,funded_amnt,1428
2,funded_amnt_inv,1835
4,int_rate,570
5,installment,28433
...,...,...
99,sec_app_open_acc,47
100,sec_app_revol_util,999
101,sec_app_open_act_il,29
102,sec_app_num_rev_accts,56


In [207]:
for v in df3.columns:
    print(v)

index
values


In [208]:
df = df.replace({True: 'True', False: 'False','TRUE':'True','FALSE':'False'})
df

,loan_amnt,funded_amnt,funded_amnt_inv,term,int_rate,installment,grade,sub_grade,emp_title,emp_length,...,sec_app_mort_acc,sec_app_open_acc,sec_app_revol_util,sec_app_open_act_il,sec_app_num_rev_accts,sec_app_chargeoff_within_12_mths,sec_app_collections_12_mths_ex_med,sec_app_mths_since_last_major_derog,disbursement_method,target
0,35000,35000,35000.0,36 months,12.12,1164.51,B,B3,Legacy Physicians Group,1,...,Missing,Missing,Missing,Missing,Missing,Missing,Missing,Missing,Cash,0
1,30000,30000,30000.0,60 months,10.75,648.54,B,B4,Director of nursing,2,...,Missing,Missing,Missing,Missing,Missing,Missing,Missing,Missing,Cash,0
2,15000,15000,15000.0,36 months,7.49,466.53,A,A4,Partner,5,...,Missing,Missing,Missing,Missing,Missing,Missing,Missing,Missing,Cash,0
3,24000,24000,24000.0,60 months,21.15,651.31,E,E2,Einstein Bros. Bagels,5,...,Missing,Missing,Missing,Missing,Missing,Missing,Missing,Missing,Cash,0
4,14400,14400,14400.0,36 months,8.59,455.18,A,A5,Nurse practitioner,3,...,Missing,Missing,Missing,Missing,Missing,Missing,Missing,Missing,Cash,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
99995,4000,4000,4000.0,36 months,13.49,135.73,C,C2,Center Based Special Education Teacher,5,...,Missing,Missing,Missing,Missing,Missing,Missing,Missing,Missing,Cash,0
99996,35000,35000,35000.0,60 months,15.59,843.53,D,D1,Senior Director,7,...,Missing,Missing,Missing,Missing,Missing,Missing,Missing,Missing,Cash,0
99997,20000,20000,20000.0,60 months,15.49,480.96,C,C4,Registered Dental Hygienist,2,...,Missing,Missing,Missing,Missing,Missing,Missing,Missing,Missing,DirectPay,0
99998,16000,16000,16000.0,36 months,11.06,524.28,B,B3,Cash Management Manager,10+,...,Missing,Missing,Missing,Missing,Missing,Missing,Missing,Missing,Cash,0


In [209]:
for v in df3['index']:
    print(v)

loan_amnt
funded_amnt
funded_amnt_inv
int_rate
installment
sub_grade
emp_title
emp_length
annual_inc
issue_d
desc
purpose
title
zip_code
addr_state
dti
delinq_2yrs
earliest_cr_line
inq_last_6mths
mths_since_last_delinq
mths_since_last_record
open_acc
pub_rec
revol_bal
revol_util
total_acc
mths_since_last_major_derog
annual_inc_joint
dti_joint
tot_coll_amt
tot_cur_bal
open_acc_6m
open_act_il
open_il_12m
open_il_24m
mths_since_rcnt_il
total_bal_il
il_util
open_rv_12m
open_rv_24m
max_bal_bc
all_util
total_rev_hi_lim
inq_fi
total_cu_tl
inq_last_12m
acc_open_past_24mths
avg_cur_bal
bc_open_to_buy
bc_util
delinq_amnt
mo_sin_old_il_acct
mo_sin_old_rev_tl_op
mo_sin_rcnt_rev_tl_op
mo_sin_rcnt_tl
mort_acc
mths_since_recent_bc
mths_since_recent_bc_dlq
mths_since_recent_inq
mths_since_recent_revol_delinq
num_accts_ever_120_pd
num_actv_bc_tl
num_actv_rev_tl
num_bc_sats
num_bc_tl
num_il_tl
num_op_rev_tl
num_rev_accts
num_rev_tl_bal_gt_0
num_sats
num_tl_90g_dpd_24m
num_tl_op_past_12m
pct_tl_nvr_dlq
p

## 4. Feature Engineering – Weight of Evidence (WOE) & Information Value (IV)

This is the **most critical step** in credit risk modeling.

### What is WOE?
**Weight of Evidence** converts categorical/binned features into a numerical score that shows how strongly each bin predicts default.

```
WOE = ln(Distribution of Good / Distribution of Bad)
```

- **Positive WOE** → More non-defaulters (good sign)
- **Negative WOE** → More defaulters (bad sign)

### What is IV?
**Information Value** measures the overall predictive power of a feature:

| IV Range | Predictive Power |
| --- | --- |
| < 0.02 | Useless |
| 0.02 – 0.10 | Weak |
| 0.10 – 0.30 | Medium |
| 0.30 – 0.50 | Strong |
| > 0.50 | Suspicious (possible overfit) |

We keep features with **IV between 0.10 and 0.50** for modeling.

In [233]:
dim_col = ["loan_amnt",
"funded_amnt",
"funded_amnt_inv",
"int_rate",
"installment",
"emp_title",
"emp_length",
"annual_inc",
"title",
"dti",
"delinq_2yrs","inq_last_6mths",
"mths_since_last_delinq",
"mths_since_last_record",
"open_acc",
"pub_rec",
"revol_bal",
"revol_util",
"total_acc",
"mths_since_last_major_derog",
"annual_inc_joint",
"dti_joint",
"tot_coll_amt",
"tot_cur_bal",
"open_acc_6m",
"open_act_il",
"open_il_12m",
"open_il_24m",
"mths_since_rcnt_il",
"total_bal_il",
"il_util",
"open_rv_12m",
"open_rv_24m",
"max_bal_bc",
"all_util",
"total_rev_hi_lim",
"inq_fi",
"total_cu_tl",
"inq_last_12m",
"acc_open_past_24mths",
"avg_cur_bal",
"bc_open_to_buy",
"bc_util",
"delinq_amnt",
"mo_sin_old_il_acct",
"mo_sin_old_rev_tl_op",
"mo_sin_rcnt_rev_tl_op",
"mo_sin_rcnt_tl",
"mort_acc",
"mths_since_recent_bc",
"mths_since_recent_bc_dlq",
"mths_since_recent_inq",
"mths_since_recent_revol_delinq",
"num_accts_ever_120_pd",
"num_actv_bc_tl",
"num_actv_rev_tl",
"num_bc_sats",
"num_bc_tl",
"num_il_tl",
"num_op_rev_tl",
"num_rev_accts",
"num_rev_tl_bal_gt_0",
"num_sats",
"num_tl_90g_dpd_24m",
"num_tl_op_past_12m",
"pct_tl_nvr_dlq",
"percent_bc_gt_75",
"tax_liens",
"tot_hi_cred_lim",
"total_bal_ex_mort",
"total_bc_limit",
"total_il_high_credit_limit",
"revol_bal_joint",
"sec_app_mort_acc",
"sec_app_open_acc",
"sec_app_revol_util",
"sec_app_open_act_il",
"sec_app_num_rev_accts",
"sec_app_mths_since_last_major_derog",]

In [234]:
clean_df = df[dim_col].apply(pd.to_numeric,errors='coerce')
percentile_q = [0.10,0.20,0.30,0.40,0.50,0.60,0.70,0.80,0.90,1.0]
percentile_results = clean_df.quantile(percentile_q)
percentile_results

,loan_amnt,funded_amnt,funded_amnt_inv,int_rate,installment,emp_title,emp_length,annual_inc,title,dti,...,total_bal_ex_mort,total_bc_limit,total_il_high_credit_limit,revol_bal_joint,sec_app_mort_acc,sec_app_open_acc,sec_app_revol_util,sec_app_open_act_il,sec_app_num_rev_accts,sec_app_mths_since_last_major_derog
0.1,5000.0,5000.0,5000.0,7.26,158.569,3.0,1.0,34000.0,2013.0,7.280,...,10684.0,4300.0,11690.0,8731.4,2.0,4.0,24.09,2.0,4.0,8.0
0.2,7000.0,7000.0,6975.0,8.46,220.764,3.0,1.0,42000.0,2013.0,10.514,...,17687.8,7000.0,18355.2,13218.0,2.0,6.0,35.70,2.0,6.0,15.0
0.3,9400.0,9375.0,9275.0,10.33,276.490,3.0,2.0,50000.0,2013.0,13.170,...,24115.0,9900.0,24426.0,17446.8,2.0,8.0,44.77,2.0,8.0,22.0
0.4,10300.0,10300.0,10200.0,11.47,327.156,3.0,3.0,57531.6,2013.0,15.530,...,30778.6,12942.4,30526.2,21739.8,2.0,9.0,52.80,3.0,9.0,29.0
0.5,12800.0,12800.0,12750.0,12.62,377.890,3.0,3.0,65000.0,2013.0,17.890,...,37889.5,16500.0,37438.0,26806.0,3.0,11.0,60.60,3.0,11.0,37.0
0.6,15000.0,15000.0,15000.0,13.67,458.860,3.0,4.0,75000.0,1662002.8,20.350,...,46344.0,21000.0,45857.8,32532.2,3.0,12.0,67.44,4.0,13.0,43.0
0.7,19350.0,19300.0,19250.0,15.02,537.030,3.0,5.0,85000.0,3321992.6,23.060,...,57206.3,26800.0,56574.6,39840.6,4.0,14.0,74.50,4.0,15.0,51.0
0.8,23100.0,23030.0,23000.0,16.99,653.110,3.0,6.0,100000.0,4981982.4,26.260,...,73199.0,35400.0,72000.0,50382.0,4.0,17.0,82.72,6.0,18.0,61.0
0.9,30000.0,30000.0,30000.0,19.52,828.230,3.0,8.0,130000.0,6641972.2,30.590,...,103023.5,50900.0,99103.2,68296.6,5.0,20.0,91.51,8.0,23.0,70.0
1.0,40000.0,40000.0,40000.0,30.99,1715.420,3.0,9.0,9522972.0,8301962.0,999.000,...,1550157.0,571500.0,1548183.0,279083.0,16.0,56.0,434.30,34.0,71.0,117.0


In [235]:
nan_cols = percentile_results.columns[percentile_results.isna().any()]
nan_cols

Index([], dtype='object')

In [236]:
df['purpose'].unique()

array(['debt_consolidation', 'credit_card', 'small_business', 'other',
       'major_purchase', 'home_improvement', 'car', 'house', 'vacation',
       'moving', 'medical', 'renewable_energy', 'wedding', 'educational'],
      dtype=object)

In [239]:
new_df = pd.DataFrame()

for col in dim_col:
    # Create bins
    binned = pd.cut(pd.to_numeric(df[col], errors='coerce'), bins=5)
    
    # Convert to string + handle missing
    binned = binned.astype(object).fillna('Missing')
    
    # Assign to new_df with original column name
    new_df[col] = binned

new_df

,loan_amnt,funded_amnt,funded_amnt_inv,int_rate,installment,emp_title,emp_length,annual_inc,title,dti,...,total_bal_ex_mort,total_bc_limit,total_il_high_credit_limit,revol_bal_joint,sec_app_mort_acc,sec_app_open_acc,sec_app_revol_util,sec_app_open_act_il,sec_app_num_rev_accts,sec_app_mths_since_last_major_derog
0,"(32160.0, 40000.0]","(32160.0, 40000.0]","(32000.0, 40000.0]","(10.446, 15.582]","(1037.752, 1376.586]",Missing,"(0.992, 2.6]","(-9522.972, 1904594.4]",Missing,"(-2.0, 199.0]",...,"(-1548.155, 310033.0]","(-471.4, 114380.0]","(-1271.907, 309857.4]",Missing,Missing,Missing,Missing,Missing,Missing,Missing
1,"(24320.0, 32160.0]","(24320.0, 32160.0]","(24000.0, 32000.0]","(10.446, 15.582]","(360.084, 698.918]",Missing,"(0.992, 2.6]","(-9522.972, 1904594.4]",Missing,"(-2.0, 199.0]",...,"(-1548.155, 310033.0]","(-471.4, 114380.0]","(-1271.907, 309857.4]",Missing,Missing,Missing,Missing,Missing,Missing,Missing
2,"(8640.0, 16480.0]","(8640.0, 16480.0]","(8000.0, 16000.0]","(5.284, 10.446]","(360.084, 698.918]",Missing,"(4.2, 5.8]","(-9522.972, 1904594.4]",Missing,"(-2.0, 199.0]",...,"(-1548.155, 310033.0]","(-471.4, 114380.0]","(-1271.907, 309857.4]",Missing,Missing,Missing,Missing,Missing,Missing,Missing
3,"(16480.0, 24320.0]","(16480.0, 24320.0]","(16000.0, 24000.0]","(20.718, 25.854]","(360.084, 698.918]",Missing,"(4.2, 5.8]","(-9522.972, 1904594.4]",Missing,"(-2.0, 199.0]",...,"(-1548.155, 310033.0]","(-471.4, 114380.0]","(-1271.907, 309857.4]",Missing,Missing,Missing,Missing,Missing,Missing,Missing
4,"(8640.0, 16480.0]","(8640.0, 16480.0]","(8000.0, 16000.0]","(5.284, 10.446]","(360.084, 698.918]",Missing,"(2.6, 4.2]","(-9522.972, 1904594.4]",Missing,"(-2.0, 199.0]",...,"(-1548.155, 310033.0]","(-471.4, 114380.0]","(-1271.907, 309857.4]",Missing,Missing,Missing,Missing,Missing,Missing,Missing
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
99995,"(760.8, 8640.0]","(760.8, 8640.0]","(-40.0, 8000.0]","(10.446, 15.582]","(19.556, 360.084]",Missing,"(4.2, 5.8]","(-9522.972, 1904594.4]",Missing,"(-2.0, 199.0]",...,"(-1548.155, 310033.0]","(-471.4, 114380.0]","(-1271.907, 309857.4]",Missing,Missing,Missing,Missing,Missing,Missing,Missing
99996,"(32160.0, 40000.0]","(32160.0, 40000.0]","(32000.0, 40000.0]","(15.582, 20.718]","(698.918, 1037.752]",Missing,"(5.8, 7.4]","(-9522.972, 1904594.4]",Missing,"(-2.0, 199.0]",...,"(-1548.155, 310033.0]","(-471.4, 114380.0]","(-1271.907, 309857.4]",Missing,Missing,Missing,Missing,Missing,Missing,Missing
99997,"(16480.0, 24320.0]","(16480.0, 24320.0]","(16000.0, 24000.0]","(10.446, 15.582]","(360.084, 698.918]",Missing,"(0.992, 2.6]","(-9522.972, 1904594.4]",Missing,"(-2.0, 199.0]",...,"(-1548.155, 310033.0]","(-471.4, 114380.0]","(-1271.907, 309857.4]",Missing,Missing,Missing,Missing,Missing,Missing,Missing
99998,"(8640.0, 16480.0]","(8640.0, 16480.0]","(8000.0, 16000.0]","(10.446, 15.582]","(360.084, 698.918]",Missing,Missing,"(-9522.972, 1904594.4]",Missing,"(-2.0, 199.0]",...,"(-1548.155, 310033.0]","(-471.4, 114380.0]","(-1271.907, 309857.4]",Missing,Missing,Missing,Missing,Missing,Missing,Missing


In [245]:
new_df['target'] = df['target']
new_df

,loan_amnt,funded_amnt,funded_amnt_inv,int_rate,installment,emp_title,emp_length,annual_inc,title,dti,...,total_bc_limit,total_il_high_credit_limit,revol_bal_joint,sec_app_mort_acc,sec_app_open_acc,sec_app_revol_util,sec_app_open_act_il,sec_app_num_rev_accts,sec_app_mths_since_last_major_derog,target
0,"(32160.0, 40000.0]","(32160.0, 40000.0]","(32000.0, 40000.0]","(10.446, 15.582]","(1037.752, 1376.586]",Missing,"(0.992, 2.6]","(-9522.972, 1904594.4]",Missing,"(-2.0, 199.0]",...,"(-471.4, 114380.0]","(-1271.907, 309857.4]",Missing,Missing,Missing,Missing,Missing,Missing,Missing,0
1,"(24320.0, 32160.0]","(24320.0, 32160.0]","(24000.0, 32000.0]","(10.446, 15.582]","(360.084, 698.918]",Missing,"(0.992, 2.6]","(-9522.972, 1904594.4]",Missing,"(-2.0, 199.0]",...,"(-471.4, 114380.0]","(-1271.907, 309857.4]",Missing,Missing,Missing,Missing,Missing,Missing,Missing,0
2,"(8640.0, 16480.0]","(8640.0, 16480.0]","(8000.0, 16000.0]","(5.284, 10.446]","(360.084, 698.918]",Missing,"(4.2, 5.8]","(-9522.972, 1904594.4]",Missing,"(-2.0, 199.0]",...,"(-471.4, 114380.0]","(-1271.907, 309857.4]",Missing,Missing,Missing,Missing,Missing,Missing,Missing,0
3,"(16480.0, 24320.0]","(16480.0, 24320.0]","(16000.0, 24000.0]","(20.718, 25.854]","(360.084, 698.918]",Missing,"(4.2, 5.8]","(-9522.972, 1904594.4]",Missing,"(-2.0, 199.0]",...,"(-471.4, 114380.0]","(-1271.907, 309857.4]",Missing,Missing,Missing,Missing,Missing,Missing,Missing,0
4,"(8640.0, 16480.0]","(8640.0, 16480.0]","(8000.0, 16000.0]","(5.284, 10.446]","(360.084, 698.918]",Missing,"(2.6, 4.2]","(-9522.972, 1904594.4]",Missing,"(-2.0, 199.0]",...,"(-471.4, 114380.0]","(-1271.907, 309857.4]",Missing,Missing,Missing,Missing,Missing,Missing,Missing,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
99995,"(760.8, 8640.0]","(760.8, 8640.0]","(-40.0, 8000.0]","(10.446, 15.582]","(19.556, 360.084]",Missing,"(4.2, 5.8]","(-9522.972, 1904594.4]",Missing,"(-2.0, 199.0]",...,"(-471.4, 114380.0]","(-1271.907, 309857.4]",Missing,Missing,Missing,Missing,Missing,Missing,Missing,0
99996,"(32160.0, 40000.0]","(32160.0, 40000.0]","(32000.0, 40000.0]","(15.582, 20.718]","(698.918, 1037.752]",Missing,"(5.8, 7.4]","(-9522.972, 1904594.4]",Missing,"(-2.0, 199.0]",...,"(-471.4, 114380.0]","(-1271.907, 309857.4]",Missing,Missing,Missing,Missing,Missing,Missing,Missing,0
99997,"(16480.0, 24320.0]","(16480.0, 24320.0]","(16000.0, 24000.0]","(10.446, 15.582]","(360.084, 698.918]",Missing,"(0.992, 2.6]","(-9522.972, 1904594.4]",Missing,"(-2.0, 199.0]",...,"(-471.4, 114380.0]","(-1271.907, 309857.4]",Missing,Missing,Missing,Missing,Missing,Missing,Missing,0
99998,"(8640.0, 16480.0]","(8640.0, 16480.0]","(8000.0, 16000.0]","(10.446, 15.582]","(360.084, 698.918]",Missing,Missing,"(-9522.972, 1904594.4]",Missing,"(-2.0, 199.0]",...,"(-471.4, 114380.0]","(-1271.907, 309857.4]",Missing,Missing,Missing,Missing,Missing,Missing,Missing,0


In [246]:
final_list_new=[]

for col in dim_col:
    group = new_df.groupby(col)['target'].agg(['count','sum'])
    group.columns = ['Total','Bad']
    group['Good'] = group['Total']-group['Bad']
    total_good = group['Good'].sum()
    total_bad = group['Bad'].sum()
    group['Good Pct'] = (group['Good']/total_good).replace(0, 0.0001)
    group['Bad Pct'] = (group['Bad']/total_bad).replace(0, 0.0001)
    group['WOE'] = np.log(group['Good Pct']/group['Bad Pct'])
    group['IV'] = (group['Good Pct']-group['Bad Pct'])*group['WOE']
    group['dimension'] = col
    group['category'] = group.index
    final_list_new.append(group.reset_index(drop=True))
df_iv1 = pd.concat(final_list_new, ignore_index=True)
df_iv1

,Total,Bad,Good,Good Pct,Bad Pct,WOE,IV,dimension,category
0,28178,2819,25359,0.286640,0.244493,0.159040,0.006703,loan_amnt,"(760.8, 8640.0]"
1,36173,4329,31844,0.359941,0.375455,-0.042199,0.000655,loan_amnt,"(8640.0, 16480.0]"
2,18617,2378,16239,0.183554,0.206245,-0.116555,0.002645,loan_amnt,"(16480.0, 24320.0]"
3,10527,1284,9243,0.104476,0.111362,-0.063825,0.000439,loan_amnt,"(24320.0, 32160.0]"
4,6505,720,5785,0.065389,0.062446,0.046061,0.000136,loan_amnt,"(32160.0, 40000.0]"
...,...,...,...,...,...,...,...,...,...
445,508,24,484,0.005471,0.002082,0.966320,0.003275,sec_app_mths_since_last_major_derog,"(25.0, 48.0]"
446,370,14,356,0.004024,0.001214,1.198162,0.003367,sec_app_mths_since_last_major_derog,"(48.0, 71.0]"
447,118,0,118,0.001334,0.000100,2.590606,0.003196,sec_app_mths_since_last_major_derog,"(71.0, 94.0]"
448,13,0,13,0.000147,0.000100,0.384871,0.000018,sec_app_mths_since_last_major_derog,"(94.0, 117.0]"


In [247]:
extra_cols = [
    'term','grade',
    "home_ownership","verification_status","pymnt_plan",
    "initial_list_status","collections_12_mths_ex_med",
    "policy_code","application_type","verification_status_joint",
    "acc_now_delinq","chargeoff_within_12_mths","num_tl_120dpd_2m",
    "num_tl_30dpd","pub_rec_bankruptcies","sec_app_inq_last_6mths",
    "sec_app_chargeoff_within_12_mths",
    "sec_app_collections_12_mths_ex_med",
    "disbursement_method",'sub_grade','purpose'
]

In [248]:
df_final = pd.concat(
    [new_df, df[extra_cols], df['target']],
    axis=1
)

In [249]:
df_final

,loan_amnt,funded_amnt,funded_amnt_inv,int_rate,installment,emp_title,emp_length,annual_inc,title,dti,...,num_tl_120dpd_2m,num_tl_30dpd,pub_rec_bankruptcies,sec_app_inq_last_6mths,sec_app_chargeoff_within_12_mths,sec_app_collections_12_mths_ex_med,disbursement_method,sub_grade,purpose,target
0,"(32160.0, 40000.0]","(32160.0, 40000.0]","(32000.0, 40000.0]","(10.446, 15.582]","(1037.752, 1376.586]",Missing,"(0.992, 2.6]","(-9522.972, 1904594.4]",Missing,"(-2.0, 199.0]",...,False,False,False,Missing,Missing,Missing,Cash,B3,debt_consolidation,0
1,"(24320.0, 32160.0]","(24320.0, 32160.0]","(24000.0, 32000.0]","(10.446, 15.582]","(360.084, 698.918]",Missing,"(0.992, 2.6]","(-9522.972, 1904594.4]",Missing,"(-2.0, 199.0]",...,False,False,False,Missing,Missing,Missing,Cash,B4,credit_card,0
2,"(8640.0, 16480.0]","(8640.0, 16480.0]","(8000.0, 16000.0]","(5.284, 10.446]","(360.084, 698.918]",Missing,"(4.2, 5.8]","(-9522.972, 1904594.4]",Missing,"(-2.0, 199.0]",...,False,False,False,Missing,Missing,Missing,Cash,A4,small_business,0
3,"(16480.0, 24320.0]","(16480.0, 24320.0]","(16000.0, 24000.0]","(20.718, 25.854]","(360.084, 698.918]",Missing,"(4.2, 5.8]","(-9522.972, 1904594.4]",Missing,"(-2.0, 199.0]",...,False,False,False,Missing,Missing,Missing,Cash,E2,debt_consolidation,0
4,"(8640.0, 16480.0]","(8640.0, 16480.0]","(8000.0, 16000.0]","(5.284, 10.446]","(360.084, 698.918]",Missing,"(2.6, 4.2]","(-9522.972, 1904594.4]",Missing,"(-2.0, 199.0]",...,False,False,False,Missing,Missing,Missing,Cash,A5,debt_consolidation,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
99995,"(760.8, 8640.0]","(760.8, 8640.0]","(-40.0, 8000.0]","(10.446, 15.582]","(19.556, 360.084]",Missing,"(4.2, 5.8]","(-9522.972, 1904594.4]",Missing,"(-2.0, 199.0]",...,False,False,False,Missing,Missing,Missing,Cash,C2,debt_consolidation,0
99996,"(32160.0, 40000.0]","(32160.0, 40000.0]","(32000.0, 40000.0]","(15.582, 20.718]","(698.918, 1037.752]",Missing,"(5.8, 7.4]","(-9522.972, 1904594.4]",Missing,"(-2.0, 199.0]",...,False,False,False,Missing,Missing,Missing,Cash,D1,debt_consolidation,0
99997,"(16480.0, 24320.0]","(16480.0, 24320.0]","(16000.0, 24000.0]","(10.446, 15.582]","(360.084, 698.918]",Missing,"(0.992, 2.6]","(-9522.972, 1904594.4]",Missing,"(-2.0, 199.0]",...,False,False,False,Missing,Missing,Missing,DirectPay,C4,debt_consolidation,0
99998,"(8640.0, 16480.0]","(8640.0, 16480.0]","(8000.0, 16000.0]","(10.446, 15.582]","(360.084, 698.918]",Missing,Missing,"(-9522.972, 1904594.4]",Missing,"(-2.0, 199.0]",...,False,False,False,Missing,Missing,Missing,Cash,B3,debt_consolidation,0


In [253]:
combined_df = pd.concat([df_iv,df_iv1])
combined_df

,Total,Bad,Good,Good Pct,Bad Pct,WOE,IV,dimension,category
0,3807,61,3746,0.042342,0.005291,2.079859,0.077062,sub_grade,A1
1,3079,61,3018,0.034113,0.005291,1.863765,0.053719,sub_grade,A2
2,3192,99,3093,0.034961,0.008586,1.404066,0.037032,sub_grade,A3
3,4334,168,4166,0.047089,0.014571,1.173036,0.038146,sub_grade,A4
4,4868,262,4606,0.052063,0.022723,0.829059,0.024324,sub_grade,A5
...,...,...,...,...,...,...,...,...,...
445,508,24,484,0.005471,0.002082,0.966320,0.003275,sec_app_mths_since_last_major_derog,"(25.0, 48.0]"
446,370,14,356,0.004024,0.001214,1.198162,0.003367,sec_app_mths_since_last_major_derog,"(48.0, 71.0]"
447,118,0,118,0.001334,0.000100,2.590606,0.003196,sec_app_mths_since_last_major_derog,"(71.0, 94.0]"
448,13,0,13,0.000147,0.000100,0.384871,0.000018,sec_app_mths_since_last_major_derog,"(94.0, 117.0]"


In [254]:
iv_summary = (
    combined_df.groupby("dimension")["IV"]
      .sum()
      .reset_index()
      .sort_values("IV", ascending=False))
iv_summary

,dimension,IV
84,sub_grade,0.500162
20,grade,0.455921
28,int_rate,0.361562
2,all_util,0.196247
30,max_bal_bc,0.161224
...,...,...
71,pymnt_plan,0.000314
12,delinq_amnt,0.000261
87,title,0.000117
17,emp_title,0.000115


In [255]:
filter_iv = iv_summary[(iv_summary['IV']>=0.1) & (iv_summary['IV']<=0.5)]
filter_iv

,dimension,IV
20,grade,0.455921
28,int_rate,0.361562
2,all_util,0.196247
30,max_bal_bc,0.161224
39,mths_since_rcnt_il,0.156943
93,total_bal_il,0.129105
22,il_util,0.114672


In [266]:
keep_cols = [
    'grade',
    'int_rate',
    'all_util',
    'max_bal_bc',
    'mths_since_rcnt_il',
    'total_bal_il',
    'il_util','target'
]

In [271]:
target_dataset = df_final[keep_cols].copy()
target_dataset

,grade,int_rate,all_util,max_bal_bc,mths_since_rcnt_il,total_bal_il,il_util,target,target
0,B,"(10.446, 15.582]",Missing,Missing,Missing,Missing,Missing,0,0
1,B,"(10.446, 15.582]","(41.0, 80.0]","(-247.998, 50001.6]","(1.514, 99.2]","(-1463.395, 293282.0]","(1.444, 113.2]",0,0
2,A,"(5.284, 10.446]","(1.805, 41.0]","(-247.998, 50001.6]","(1.514, 99.2]","(-1463.395, 293282.0]","(1.444, 113.2]",0,0
3,E,"(20.718, 25.854]",Missing,Missing,Missing,Missing,Missing,0,0
4,A,"(5.284, 10.446]","(80.0, 119.0]","(-247.998, 50001.6]","(1.514, 99.2]","(-1463.395, 293282.0]","(1.444, 113.2]",0,0
...,...,...,...,...,...,...,...,...,...
99995,C,"(10.446, 15.582]","(80.0, 119.0]","(-247.998, 50001.6]","(1.514, 99.2]","(-1463.395, 293282.0]","(1.444, 113.2]",0,0
99996,D,"(15.582, 20.718]",Missing,Missing,Missing,Missing,Missing,0,0
99997,C,"(10.446, 15.582]","(80.0, 119.0]","(-247.998, 50001.6]","(1.514, 99.2]","(-1463.395, 293282.0]","(1.444, 113.2]",0,0
99998,B,"(10.446, 15.582]","(41.0, 80.0]","(-247.998, 50001.6]","(1.514, 99.2]","(-1463.395, 293282.0]","(1.444, 113.2]",0,0


In [275]:
print(target_dataset.columns)

Index(['grade', 'int_rate', 'all_util', 'max_bal_bc', 'mths_since_rcnt_il',
       'total_bal_il', 'il_util', 'target'],
      dtype='object')


In [274]:
target_dataset = target_dataset.loc[:, ~target_dataset.columns.duplicated()]

In [276]:
target_dataset

,grade,int_rate,all_util,max_bal_bc,mths_since_rcnt_il,total_bal_il,il_util,target
0,B,"(10.446, 15.582]",Missing,Missing,Missing,Missing,Missing,0
1,B,"(10.446, 15.582]","(41.0, 80.0]","(-247.998, 50001.6]","(1.514, 99.2]","(-1463.395, 293282.0]","(1.444, 113.2]",0
2,A,"(5.284, 10.446]","(1.805, 41.0]","(-247.998, 50001.6]","(1.514, 99.2]","(-1463.395, 293282.0]","(1.444, 113.2]",0
3,E,"(20.718, 25.854]",Missing,Missing,Missing,Missing,Missing,0
4,A,"(5.284, 10.446]","(80.0, 119.0]","(-247.998, 50001.6]","(1.514, 99.2]","(-1463.395, 293282.0]","(1.444, 113.2]",0
...,...,...,...,...,...,...,...,...
99995,C,"(10.446, 15.582]","(80.0, 119.0]","(-247.998, 50001.6]","(1.514, 99.2]","(-1463.395, 293282.0]","(1.444, 113.2]",0
99996,D,"(15.582, 20.718]",Missing,Missing,Missing,Missing,Missing,0
99997,C,"(10.446, 15.582]","(80.0, 119.0]","(-247.998, 50001.6]","(1.514, 99.2]","(-1463.395, 293282.0]","(1.444, 113.2]",0
99998,B,"(10.446, 15.582]","(41.0, 80.0]","(-247.998, 50001.6]","(1.514, 99.2]","(-1463.395, 293282.0]","(1.444, 113.2]",0


In [277]:
combined_df

,Total,Bad,Good,Good Pct,Bad Pct,WOE,IV,dimension,category
0,3807,61,3746,0.042342,0.005291,2.079859,0.077062,sub_grade,A1
1,3079,61,3018,0.034113,0.005291,1.863765,0.053719,sub_grade,A2
2,3192,99,3093,0.034961,0.008586,1.404066,0.037032,sub_grade,A3
3,4334,168,4166,0.047089,0.014571,1.173036,0.038146,sub_grade,A4
4,4868,262,4606,0.052063,0.022723,0.829059,0.024324,sub_grade,A5
...,...,...,...,...,...,...,...,...,...
445,508,24,484,0.005471,0.002082,0.966320,0.003275,sec_app_mths_since_last_major_derog,"(25.0, 48.0]"
446,370,14,356,0.004024,0.001214,1.198162,0.003367,sec_app_mths_since_last_major_derog,"(48.0, 71.0]"
447,118,0,118,0.001334,0.000100,2.590606,0.003196,sec_app_mths_since_last_major_derog,"(71.0, 94.0]"
448,13,0,13,0.000147,0.000100,0.384871,0.000018,sec_app_mths_since_last_major_derog,"(94.0, 117.0]"


In [278]:
woe_dict = {}

for col in target_dataset.columns:
    if col == 'target':
        continue
    
    temp = combined_df[combined_df['dimension'] == col]
    woe_dict[col] = dict(zip(temp['category'], temp['WOE']))

In [279]:
df_woe = target_dataset.copy()

for col in woe_dict:
    df_woe[col] = df_woe[col].map(woe_dict[col])

In [280]:
df_woe

,grade,int_rate,all_util,max_bal_bc,mths_since_rcnt_il,total_bal_il,il_util,target
0,0.446486,0.038082,-0.445398,-0.424182,-0.405837,-0.343865,-0.311422,0
1,0.446486,0.038082,0.349483,0.384177,0.394037,0.375038,0.373897,0
2,1.316254,0.983267,0.784951,0.384177,0.394037,0.375038,0.373897,0
3,-0.956901,-0.855502,-0.445398,-0.424182,-0.405837,-0.343865,-0.311422,0
4,1.316254,0.983267,0.015740,0.384177,0.394037,0.375038,0.373897,0
...,...,...,...,...,...,...,...,...
99995,-0.110526,0.038082,0.015740,0.384177,0.394037,0.375038,0.373897,0
99996,-0.539320,-0.565173,-0.445398,-0.424182,-0.405837,-0.343865,-0.311422,0
99997,-0.110526,0.038082,0.015740,0.384177,0.394037,0.375038,0.373897,0
99998,0.446486,0.038082,0.349483,0.384177,0.394037,0.375038,0.373897,0


### 5.1 Train-Test Split

We split the WOE-encoded dataset into **70% training** and **30% testing** sets using `train_test_split` with `random_state=42` for reproducibility.

## 5. Machine Learning – Model Building & Evaluation

We train and compare **three classification models** to predict loan default:

| # | Model | Type | Why? |
| --- | --- | --- | --- |
| 1 | **Logistic Regression** | Linear | Industry standard, interpretable |
| 2 | **Random Forest** | Ensemble (Bagging) | Handles non-linear patterns |
| 3 | **XGBoost** | Ensemble (Boosting) | State-of-the-art, best accuracy |

**Evaluation Metrics Used:**
- **ROC-AUC Score** — Measures model's ability to distinguish defaulters from non-defaulters (higher = better)
- **Confusion Matrix** — Shows True Positives, False Positives, True Negatives, False Negatives
- **Classification Report** — Precision, Recall, F1-Score

In [282]:
X = df_woe.drop(columns=['target'])
y = df_woe['target']

In [283]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y
)

In [467]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, confusion_matrix, classification_report, roc_curve

# Train model
lr_model = LogisticRegression(

    class_weight='balanced',
    max_iter=1000,
    random_state=42
)

lr_model.fit(X_train, y_train)

,"penalty penalty: {'l1', 'l2', 'elasticnet', None}, default='l2'Specify the norm of the penalty:- `None`: no penalty is added;- `'l2'`: add a L2 penalty term and it is the default choice;- `'l1'`: add a L1 penalty term;- `'elasticnet'`: both L1 and L2 penalty terms are added... warning:: Some penalties may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionadded:: 0.19 l1 penalty with SAGA solver (allowing 'multinomial' + L1).. deprecated:: 1.8 `penalty` was deprecated in version 1.8 and will be removed in 1.10. Use `l1_ratio` instead. `l1_ratio=0` for `penalty='l2'`, `l1_ratio=1` for `penalty='l1'` and `l1_ratio` set to any float between 0 and 1 for `'penalty='elasticnet'`.",'deprecated'
,"C C: float, default=1.0Inverse of regularization strength; must be a positive float.Like in support vector machines, smaller values specify strongerregularization. `C=np.inf` results in unpenalized logistic regression.For a visual example on the effect of tuning the `C` parameterwith an L1 penalty, see::ref:`sphx_glr_auto_examples_linear_model_plot_logistic_path.py`.",1.0
,"l1_ratio l1_ratio: float, default=0.0The Elastic-Net mixing parameter, with `0 <= l1_ratio <= 1`. Setting`l1_ratio=1` gives a pure L1-penalty, setting `l1_ratio=0` a pure L2-penalty.Any value between 0 and 1 gives an Elastic-Net penalty of the form`l1_ratio * L1 + (1 - l1_ratio) * L2`... warning:: Certain values of `l1_ratio`, i.e. some penalties, may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionchanged:: 1.8 Default value changed from None to 0.0... deprecated:: 1.8 `None` is deprecated and will be removed in version 1.10. Always use `l1_ratio` to specify the penalty type.",0.0
,"dual dual: bool, default=FalseDual (constrained) or primal (regularized, see also:ref:`this equation `) formulation. Dual formulationis only implemented for l2 penalty with liblinear solver. Prefer `dual=False`when n_samples > n_features.",False
,"tol tol: float, default=1e-4Tolerance for stopping criteria.",0.0001
,"fit_intercept fit_intercept: bool, default=TrueSpecifies if a constant (a.k.a. bias or intercept) should beadded to the decision function.",True
,"intercept_scaling intercept_scaling: float, default=1Useful only when the solver `liblinear` is usedand `self.fit_intercept` is set to `True`. In this case, `x` becomes`[x, self.intercept_scaling]`,i.e. a ""synthetic"" feature with constant value equal to`intercept_scaling` is appended to the instance vector.The intercept becomes``intercept_scaling * synthetic_feature_weight``... note:: The synthetic feature weight is subject to L1 or L2 regularization as all other features. To lessen the effect of regularization on synthetic feature weight (and therefore on the intercept) `intercept_scaling` has to be increased.",1
,"class_weight class_weight: dict or 'balanced', default=NoneWeights associated with classes in the form ``{class_label: weight}``.If not given, all classes are supposed to have weight one.The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``.Note that these weights will be multiplied with sample_weight (passedthrough the fit method) if sample_weight is specified... versionadded:: 0.17 *class_weight='balanced'*",'balanced'
,"random_state random_state: int, RandomState instance, default=NoneUsed when ``solver`` == 'sag', 'saga' or 'liblinear' to shuffle thedata. See :term:`Glossary ` for details.",42
,"solver solver: {'lbfgs', 'liblinear', 'newton-cg', 'newton-cholesky', 'sag', 'saga'}, default='lbfgs'Algorithm to use in the optimization problem. Default is 'lbfgs'.To choose a solver, you might want to consider the following aspects:- 'lbfgs' is a good default solver because it works reasonably well for a wide class of problems.- For :term:

In [468]:
# Predict probabilities
y_pred_prob = lr_model.predict_proba(X_test)[:, 1]

# ROC AUC
roc_auc_LR = roc_auc_score(y_test, y_pred_prob)
print("ROC AUC:", roc_auc_LR)

# KS from ROC
fpr, tpr, thresholds = roc_curve(y_test, y_pred_prob)
ks_value = max(tpr - fpr)
print("KS Score:", ks_value)

# 🔴 MANUAL THRESHOLD (you control this)
threshold = 0.55   # <-- change this value

y_pred = (y_pred_prob >= threshold).astype(int)

# Confusion Matrix
cm_LR = confusion_matrix(y_test, y_pred)
print("Confusion Matrix:\n", cm_LR)

# Classification Report
report_lr = classification_report(y_test, y_pred)
print("Classification Report:\n", report_lr)

ROC AUC: 0.7097320472248454
KS Score: 0.31049704211582774
Confusion Matrix:
 [[20235  6306]
 [ 1638  1821]]
Classification Report:
               precision    recall  f1-score   support

           0       0.93      0.76      0.84     26541
           1       0.22      0.53      0.31      3459

    accuracy                           0.74     30000
   macro avg       0.57      0.64      0.58     30000
weighted avg       0.84      0.74      0.78     30000



In [451]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score, confusion_matrix, classification_report

# Train model
rf_model = RandomForestClassifier(
    n_estimators=200,
    class_weight='balanced',
    max_depth=12,
    random_state=42
)

rf_model.fit(X_train, y_train)


,"n_estimators n_estimators: int, default=100The number of trees in the forest... versionchanged:: 0.22 The default value of ``n_estimators`` changed from 10 to 100 in 0.22.",200
,"criterion criterion: {""gini"", ""entropy"", ""log_loss""}, default=""gini""The function to measure the quality of a split. Supported criteria are""gini"" for the Gini impurity and ""log_loss"" and ""entropy"" both for theShannon information gain, see :ref:`tree_mathematical_formulation`.Note: This parameter is tree-specific.",'gini'
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",12
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",2
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",1
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: {""sqrt"", ""log2"", None}, int or float, default=""sqrt""The number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None, then `max_features=n_features`... versionchanged:: 1.1 The default of `max_features` changed from `""auto""` to `""sqrt""`.Note: the search for a split does not stop until at least onevalid partition of the node samples is found, even if it requires toeffectively inspect more than ``max_features`` features.",'sqrt'
,"max_leaf_nodes max_leaf_nodes: int, default=NoneGrow trees with ``max_leaf_nodes`` in best-first fashion.Best nodes are defined as relative reduction in impurity.If None then unlimited number of leaf nodes.",None
,"min_impurity_decrease min_impurity_decrease: float, default=0.0A node will be split if this split induces a decrease of the impuritygreater than or equal to this value.The weighted impurity decrease equation is the following:: N_t / N * (impurity - N_t_R / N_t * right_impurity - N_t_L / N_t * left_impurity)where ``N`` is the total number of samples, ``N_t`` is the number ofsamples at the current node, ``N_t_L`` is the number of samples in theleft child, and ``N_t_R`` is the number of samples in the right child.``N``, ``N_t``, ``N_t_R`` and ``N_t_L`` all refer to the weighted sum,if ``sample_weight`` is passed... versionadded:: 0.19",0.0
,"bootstrap bootstrap: bool, default=TrueWhether bootstrap samples are used when building trees. If False, thewhole dataset is used to build each tree.",True
,"oob_score oob_score: bool or callable, default=FalseWhether to use out-of-bag samples to estimate the generalization score.By default, :func:`~sklearn.metrics.accuracy_score` is used.Provide a callable with signature `metric(y

In [453]:
# Predict probabilities
y_pred_prob = rf_model.predict_proba(X_test)[:, 1]

# ROC AUC
roc_auc_RF = roc_auc_score(y_test, y_pred_prob)
print("ROC AUC:", roc_auc_RF)

# Apply threshold
threshold = 0.55
y_pred = (y_pred_prob >= threshold).astype(int)

# Confusion Matrix
cm_RF = confusion_matrix(y_test, y_pred)
print("Confusion Matrix:\n", cm_RF)

# Classification Report
report1 = classification_report(y_test, y_pred)
print("Classification Report:\n", report1)
from sklearn.metrics import roc_curve

# ROC curve
fpr, tpr, thresholds = roc_curve(y_test, y_pred_prob)

# KS = max(TPR - FPR)
ks_value = max(tpr - fpr)
print("KS Score (from ROC):", ks_value)

ROC AUC: 0.7064294716954255
Confusion Matrix:
 [[20575  5966]
 [ 1713  1746]]
Classification Report:
               precision    recall  f1-score   support

           0       0.92      0.78      0.84     26541
           1       0.23      0.50      0.31      3459

    accuracy                           0.74     30000
   macro avg       0.57      0.64      0.58     30000
weighted avg       0.84      0.74      0.78     30000

KS Score (from ROC): 0.3066784725185694


In [459]:
from xgboost import XGBClassifier
from sklearn.metrics import roc_auc_score, confusion_matrix, classification_report, roc_curve

xgb = XGBClassifier(
    objective='binary:logistic',
    eval_metric='auc',
    n_estimators=300,
    max_depth=3,              # keep smaller (you have few features)
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,     # add this (important)
    scale_pos_weight = (len(y_train[y_train==0]) / len(y_train[y_train==1])),  # imbalance fix
    random_state=42
)

xgb.fit(X_train, y_train)


,"objective objective: typing.Union[str, xgboost.sklearn._SklObjWProto, typing.Callable[[typing.Any, typing.Any], typing.Tuple[numpy.ndarray, numpy.ndarray]], NoneType]Specify the learning task and the corresponding learning objective or a customobjective function to be used.For custom objective, see :doc:`/tutorials/custom_metric_obj` and:ref:`custom-obj-metric` for more information, along with the end note forfunction signatures.",'binary:logistic'
,"base_score base_score: typing.Union[float, typing.List[float], NoneType]The initial prediction score of all instances, global bias.",None
,booster,None
,"callbacks callbacks: typing.Optional[typing.List[xgboost.callback.TrainingCallback]]List of callback functions that are applied at end of each iteration.It is possible to use predefined callbacks by using:ref:`Callback API `... note:: States in callback are not preserved during training, which means callback objects can not be reused for multiple training sessions without reinitialization or deepcopy... code-block:: python for params in parameters_grid: # be sure to (re)initialize the callbacks before each run callbacks = [xgb.callback.LearningRateScheduler(custom_rates)] reg = xgboost.XGBRegressor(**params, callbacks=callbacks) reg.fit(X, y)",None
,colsample_bylevel colsample_bylevel: typing.Optional[float]Subsample ratio of columns for each level.,None
,colsample_bynode colsample_bynode: typing.Optional[float]Subsample ratio of columns for each split.,None
,colsample_bytree colsample_bytree: typing.Optional[float]Subsample ratio of columns when constructing each tree.,0.8
,"device device: typing.Optional[str].. versionadded:: 2.0.0Device ordinal, available options are `cpu`, `cuda`, and `gpu`.",None
,"early_stopping_rounds early_stopping_rounds: typing.Optional[int].. versionadded:: 1.6.0- Activates early stopping. Validation metric needs to improve at least once in every **early_stopping_rounds** round(s) to continue training. Requires at least one item in **eval_set** in :py:meth:`fit`.- If early stopping occurs, the model will have two additional attributes: :py:attr:`best_score` and :py:attr:`best_iteration`. These are used by the :py:meth:`predict` and :py:meth:`apply` methods to determine the optimal number of trees during inference. If users want to access the full model (including trees built after early stopping), they can specify the `iteration_range` in these inference methods. In addition, other utilities like model plotting can also use the entire model.- If you prefer to discard the trees after `best_iteration`, consider using the callback function :py:class:`xgboost.callback.EarlyStopping`.- If there's more than one item in **eval_set**, the last entry will be used for early stopping. If there's more than one metric in **eval_metric**, the last metric will be used for early stopping.",None
,enable_categorical enable_categorical: boolSee the same parameter of :py:class:`DMatrix` for details.,False
,"eval_metric eval_metric: typing.Union[str, typing.List[typing.Union[str, typing.Callable]], typing.Callable, NoneType].. versionadded:: 1.6.0Metric used for monitoring the training result and early stopping. It can be astring or list of strings as names of predefined metric in XGBoost (See:doc:`/parameter`), one of the metrics in :py:mod:`sklearn.metrics`, or anyother user defined metric that looks like `sklearn.metrics`.If custom objective is also provided, then custom metric should implement thecorresponding reverse link function.Unlike the `scoring` parameter commonly used in scikit-learn, when a callableobject is provided, it's assumed to be a cost function and by default XGBoostwill minimize the result during early stopping.For advanced usage on Early stopping like directly choosing to maximize insteadof minimize, see :py:obj:`xgboost.callback.EarlyStopping`.See :doc:`/tutorials/custom_metric_obj` and :ref:`custom-obj-metric` for moreinformation... code-block:: python from sklearn.datasets import load_diabetes f

In [466]:
# Predict probabilities
y_pred_prob = xgb.predict_proba(X_test)[:, 1]

# ROC AUC
roc_auc_xgb = roc_auc_score(y_test, y_pred_prob)
print("ROC AUC:", roc_auc_xgb)

# Apply threshold
threshold = 0.55
y_pred = (y_pred_prob >= threshold).astype(int)

# Confusion Matrix
cm_xgb = confusion_matrix(y_test, y_pred)
print("Confusion Matrix:\n", cm_xgb)

# Classification Report
report_xgb = classification_report(y_test, y_pred)
print("Classification Report:\n", report_xgb)


# ROC Curve
fpr, tpr, thresholds = roc_curve(y_test, y_pred_prob)

# KS Score (important for you)
ks_xgb = max(tpr - fpr)
print("KS Score:", ks_xgb)

ROC AUC: 0.7124474944638011
Confusion Matrix:
 [[20490  6051]
 [ 1681  1778]]
Classification Report:
               precision    recall  f1-score   support

           0       0.92      0.77      0.84     26541
           1       0.23      0.51      0.32      3459

    accuracy                           0.74     30000
   macro avg       0.58      0.64      0.58     30000
weighted avg       0.84      0.74      0.78     30000

KS Score: 0.31174246015092005


## 6. Model Results – Risk Bucketing & Portfolio Analysis

After building the models, we assign each loan into **risk buckets** (Low / Medium / High) based on its predicted probability of default.

| Risk Bucket | Probability Range | Meaning |
| --- | --- | --- |
| 🟢 Low | 0 – 0.30 | Likely to repay |
| 🟡 Medium | 0.30 – 0.60 | Borderline — needs monitoring |
| 🔴 High | 0.60 – 1.00 | High chance of default |

We also export the final test dataset for dashboard monitoring.

In [487]:
df_test_summary = X_test.copy().reset_index(drop=True)
df_test_summary['target'] = y_test.reset_index(drop=True)
df_test_summary['prob'] = lr_model.predict_proba(X_test)[:,1]

df_test_summary['risk_bucket'] = pd.cut(
    df_test_summary['prob'],
    bins=[0,0.2,0.5,1],
    labels=['Low','Medium','High'],
    include_lowest=True
)

In [489]:
df_test_summary

,grade,int_rate,all_util,max_bal_bc,mths_since_rcnt_il,total_bal_il,il_util,target,prob,risk_bucket
0,-0.110526,0.038082,0.349483,0.384177,0.394037,0.375038,0.373897,0,0.434878,Medium
1,-0.110526,0.038082,0.015740,0.384177,0.394037,0.375038,0.373897,0,0.469645,Medium
2,-0.539320,-0.855502,0.349483,0.384177,0.394037,0.375038,0.373897,0,0.548086,High
3,0.446486,0.983267,-0.445398,-0.424182,-0.405837,-0.343865,-0.311422,1,0.480331,Medium
4,-0.110526,0.038082,-0.445398,-0.424182,-0.405837,-0.343865,-0.311422,1,0.620990,High
...,...,...,...,...,...,...,...,...,...,...
29995,0.446486,0.038082,0.015740,0.384177,0.394037,-0.343865,-0.311422,0,0.377847,Medium
29996,-0.956901,-0.565173,-0.445398,-0.424182,-0.405837,-0.343865,-0.311422,0,0.784523,High
29997,-0.956901,-0.565173,-0.445398,-0.424182,-0.405837,-0.343865,-0.311422,0,0.784523,High
29998,1.316254,0.983267,-0.445398,-0.424182,-0.405837,-0.343865,-0.311422,0,0.300247,Medium


In [ ]:
df_test_summary.groupby('risk_bucket')['target'].mean()

C:\Users\om\AppData\Local\Temp\ipykernel_20440\975944790.py:1: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  df_test_summary.groupby('risk_bucket')['target'].mean()


risk_bucket
Low       0.020882
Medium    0.075173
High      0.195099
Name: target, dtype: float64

In [491]:
df_test_summary['risk_bucket'].value_counts(normalize=True)

risk_bucket
Medium    0.477567
High      0.393133
Low       0.129300
Name: proportion, dtype: float64

In [492]:
df_test_summary.to_csv("final_dataset.csv", index=False)

## 7. Stress Testing & Expected Loss (EL) Estimation

In this section, we simulate **adverse economic conditions** (stress scenarios) by scaling up predicted default probabilities.

**Key Concepts:**
- **Expected Loss (EL)** = Probability of Default (PD) × Loss Given Default (LGD) × Exposure at Default (EAD)
- **Stress Factor** = A multiplier (e.g., 1.3×) applied to PD to simulate a recession
- **Risk Migration** = How loans shift from lower-risk to higher-risk buckets under stress

This helps banks understand how much **extra capital** they need to set aside for bad times.

In [507]:
stress_df = df.copy()

# Apply stress directly on probability
stress_df['stressed_prob'] = df['prob'] * 1.15

# cap at 1
stress_df['stressed_prob'] = stress_df['stressed_prob'].clip(0,1)

In [508]:
stress_df

,grade,int_rate,all_util,max_bal_bc,mths_since_rcnt_il,total_bal_il,il_util,target,prob,risk_bucket,stressed_prob
0,-0.110526,0.038082,0.349483,0.384177,0.394037,0.375038,0.373897,0,0.434878,Medium,0.500110
1,-0.110526,0.038082,0.015740,0.384177,0.394037,0.375038,0.373897,0,0.469645,Medium,0.540092
2,-0.539320,-0.855502,0.349483,0.384177,0.394037,0.375038,0.373897,0,0.548086,High,0.630299
3,0.446486,0.983267,-0.445398,-0.424182,-0.405837,-0.343865,-0.311422,1,0.480331,Medium,0.552381
4,-0.110526,0.038082,-0.445398,-0.424182,-0.405837,-0.343865,-0.311422,1,0.620990,High,0.714138
...,...,...,...,...,...,...,...,...,...,...,...
29995,0.446486,0.038082,0.015740,0.384177,0.394037,-0.343865,-0.311422,0,0.377847,Medium,0.434524
29996,-0.956901,-0.565173,-0.445398,-0.424182,-0.405837,-0.343865,-0.311422,0,0.784523,High,0.902202
29997,-0.956901,-0.565173,-0.445398,-0.424182,-0.405837,-0.343865,-0.311422,0,0.784523,High,0.902202
29998,1.316254,0.983267,-0.445398,-0.424182,-0.405837,-0.343865,-0.311422,0,0.300247,Medium,0.345284


In [509]:
print("Before:", df['prob'].mean())
print("After:", stress_df['stressed_prob'].mean())

Before: 0.4469334455132024
After: 0.5139734623401827


In [511]:
stress_df['stressed_prob'] = df['prob'] * 1.3
stress_df['stressed_prob'] = stress_df['stressed_prob'].clip(0,1)

In [512]:
stress_df['stressed_bucket'] = pd.cut(
    stress_df['stressed_prob'],
    bins=[0, 0.3, 0.6, 1],
    labels=['Low', 'Medium', 'High']
)

pd.crosstab(df['risk_bucket'], stress_df['stressed_bucket'])

stressed_bucket,Low,Medium,High
risk_bucket,,,
High,0,0,11794
Low,3879,0,0
Medium,26,11768,2533


In [513]:
medium_total = df[df['risk_bucket']=='Medium'].shape[0]
moved_to_high = 2533

print("Migration %:", moved_to_high / medium_total)

Migration %: 0.17679905074335173


In [514]:
LGD = 0.6
EAD = 100000

df['EL'] = df['prob'] * LGD * EAD
stress_df['EL'] = stress_df['stressed_prob'] * LGD * EAD

In [517]:
stress_df

,grade,int_rate,all_util,max_bal_bc,mths_since_rcnt_il,total_bal_il,il_util,target,prob,risk_bucket,stressed_prob,stressed_bucket,EL
0,-0.110526,0.038082,0.349483,0.384177,0.394037,0.375038,0.373897,0,0.434878,Medium,0.565341,Medium,33920.482611
1,-0.110526,0.038082,0.015740,0.384177,0.394037,0.375038,0.373897,0,0.469645,Medium,0.610538,High,36632.296613
2,-0.539320,-0.855502,0.349483,0.384177,0.394037,0.375038,0.373897,0,0.548086,High,0.712512,High,42750.708166
3,0.446486,0.983267,-0.445398,-0.424182,-0.405837,-0.343865,-0.311422,1,0.480331,Medium,0.624431,High,37465.852974
4,-0.110526,0.038082,-0.445398,-0.424182,-0.405837,-0.343865,-0.311422,1,0.620990,High,0.807287,High,48437.214315
...,...,...,...,...,...,...,...,...,...,...,...,...,...
29995,0.446486,0.038082,0.015740,0.384177,0.394037,-0.343865,-0.311422,0,0.377847,Medium,0.491201,Medium,29472.044440
29996,-0.956901,-0.565173,-0.445398,-0.424182,-0.405837,-0.343865,-0.311422,0,0.784523,High,1.000000,High,60000.000000
29997,-0.956901,-0.565173,-0.445398,-0.424182,-0.405837,-0.343865,-0.311422,0,0.784523,High,1.000000,High,60000.000000
29998,1.316254,0.983267,-0.445398,-0.424182,-0.405837,-0.343865,-0.311422,0,0.300247,Medium,0.390321,Medium,23419.234525


In [519]:
print("Before EL:", df['EL'].sum())
print("After EL:", stress_df['EL'].sum())

Before EL: 804480201.9237645
After EL: 1042417047.2918239


In [524]:
el_by_bucket = df.groupby('risk_bucket')['EL'].sum()

print(el_by_bucket.apply(lambda x: f"{x:,.0f}"))

risk_bucket
High      443,841,771
Low        37,496,681
Medium    323,141,750
Name: EL, dtype: object
